# Token Counting: GPT vs Hugging Face models

Every model family tokenizes text differently, so there's no single "token count" for a string — it depends which model will actually process it.

| Model family | Correct tool | Why |
|---|---|---|
| OpenAI GPT | `tiktoken` | OpenAI's own tokenizer library |
| Hugging Face models (BERT, T5, etc.) | `transformers.AutoTokenizer` | Loads the model's own tokenizer files from the Hub |

In [2]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "ensurepip", "--upgrade"])

CompletedProcess(args=['c:\\Users\\hp\\Desktop\\genai python\\genai_class\\.venv\\Scripts\\python.exe', '-m', 'ensurepip', '--upgrade'], returncode=0)

In [1]:
import os
from dotenv import load_dotenv

SAMPLE_TEXT = (
    "Large language models process text as tokens, not characters or words. "
    "Each model family (OpenAI, Anthropic, open-source models on Hugging Face) "
    "uses its own tokenizer, so the same sentence can have a different token count "
    "depending on which model will process it. This matters for cost estimation, "
    "context-window budgeting, and rate-limit planning."
)


In [1]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "transformers"])

CompletedProcess(args=['c:\\Users\\hp\\Desktop\\genai python\\genai_class\\.venv\\Scripts\\python.exe', '-m', 'pip', 'install', 'transformers'], returncode=0)

## 1. OpenAI GPT — `tiktoken`

`tiktoken` is OpenAI's own tokenizer library. It's fast, runs fully offline, and is the *correct* tool for GPT models — but only for GPT models.

In [2]:
import tiktoken

text = "hii67"

# GPT-5.4 Nano uses the o200k_base tokenizer
encoding = tiktoken.get_encoding("o200k_base")

tokens = encoding.encode(text)

print(f"Token Count: {len(tokens)}")

Token Count: 3


In [ ]:
tokens

In [3]:
text2="""[
  {
    "role": "",
    "content": "ho w arecjucjicvdx you?"
  }
]"""


tokens = encoding.encode(text2)

print(f"Token Count: {len(tokens)}")

Token Count: 27


In [4]:
import tiktoken

encoding = tiktoken.encoding_for_model("gpt-3.5-turbo")

text = "hi lol"

tokens = encoding.encode(text)

print(tokens)
print(len(tokens))

[6151, 28509]
2


## 2. Hugging Face models — `transformers.AutoTokenizer`

Open-source models on the Hub ship their own tokenizer files. `AutoTokenizer.from_pretrained(...)` downloads just those files (not the model weights) and gives you an exact tokenizer.

In [1]:
from transformers import AutoTokenizer

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained("gpt2")

text = "Hi! How can I help you today?"

# Tokenize
token_ids = tokenizer.encode(text)

print("Token IDs:", token_ids)
print("Token Count:", len(token_ids))

c:\Users\hp\Desktop\genai python\genai_class\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
c:\Users\hp\Desktop\genai python\genai_class\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\hp\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limita

Token IDs: [17250, 0, 1374, 460, 314, 1037, 345, 1909, 30]
Token Count: 9


In [2]:
import tiktoken
from transformers import AutoTokenizer

# ---------------------------------------------------
# Text to compare
# ---------------------------------------------------
text = """
Hello, I'm Bipin! 👋
Email: bipin@example.com
Price: ₹1,250.50
Python: print("Hello, World!")
"""

# ---------------------------------------------------
# OpenAI Tokenizer (GPT-5.4 Nano / GPT-4o)
# ---------------------------------------------------
openai_encoding = tiktoken.get_encoding("o200k_base")

openai_token_ids = openai_encoding.encode(text)

openai_tokens = [
    openai_encoding.decode([token])
    for token in openai_token_ids
]

# ---------------------------------------------------
# Hugging Face Tokenizer
# Change this model to compare different tokenizers
# ---------------------------------------------------
hf_model = "gpt2"

hf_tokenizer = AutoTokenizer.from_pretrained(hf_model)

hf_token_ids = hf_tokenizer.encode(text)

hf_tokens = hf_tokenizer.convert_ids_to_tokens(hf_token_ids)

# ---------------------------------------------------
# Results
# ---------------------------------------------------
print("=" * 70)
print("Original Text")
print("=" * 70)
print(text)

print("\n")
print("=" * 70)
print("OpenAI (tiktoken)")
print("=" * 70)

print("Encoding :", "o200k_base")
print("Token Count :", len(openai_token_ids))
print("Token IDs :", openai_token_ids)
print("Tokens :", openai_tokens)

print("\n")
print("=" * 70)
print(f"Hugging Face ({hf_model})")
print("=" * 70)

print("Token Count :", len(hf_token_ids))
print("Token IDs :", hf_token_ids)
print("Tokens :", hf_tokens)

print("\n")
print("=" * 70)
print("Comparison")
print("=" * 70)

print(f"OpenAI Tokens      : {len(openai_token_ids)}")
print(f"HuggingFace Tokens : {len(hf_token_ids)}")
print(f"Difference         : {abs(len(openai_token_ids)-len(hf_token_ids))}")

Original Text

Hello, I'm Bipin! 👋
Email: bipin@example.com
Price: ₹1,250.50
Python: print("Hello, World!")



OpenAI (tiktoken)
Encoding : o200k_base
Token Count : 34
Token IDs : [198, 13225, 11, 5477, 172840, 258, 0, 61138, 233, 198, 6622, 25, 54467, 258, 81309, 1136, 198, 7417, 25, 73406, 16, 11, 6911, 13, 1434, 198, 60502, 25, 2123, 568, 13225, 11, 5922, 50941]
Tokens : ['\n', 'Hello', ',', " I'm", ' Bip', 'in', '!', ' �', '�', '\n', 'Email', ':', ' bip', 'in', '@example', '.com', '\n', 'Price', ':', ' ₹', '1', ',', '250', '.', '50', '\n', 'Python', ':', ' print', '("', 'Hello', ',', ' World', '!")\n']


Hugging Face (gpt2)
Token Count : 42
Token IDs : [198, 15496, 11, 314, 1101, 347, 541, 259, 0, 50169, 233, 198, 15333, 25, 14141, 259, 31, 20688, 13, 785, 198, 18124, 25, 2343, 224, 117, 16, 11, 9031, 13, 1120, 198, 37906, 25, 3601, 7203, 15496, 11, 2159, 2474, 8, 198]
Tokens : ['Ċ', 'Hello', ',', 'ĠI', "'m", 'ĠB', 'ip', 'in', '!', 'ĠðŁĳ', 'ĭ', 'Ċ', 'Email', ':', 'Ġbip', 'in', '@',

In [3]:
text = """
OpenAI & Hugging Face are popular AI platforms.
Visit: https://huggingface.co
Today is 28/07/2026 🚀
"""

text = """
Hi! 😊
Order ID: #A12345
Total: $49.99
Math: x² + y² = z²
"""

text = """
Hello, Bipin! 👋
Email: bipin@example.com
Price: ₹999.99
Code: print("Hello")
Math: x² + y² = z²
"""